# Sentiment Analysis with Deep Learning

This notebook demonstrates building and training neural networks for sentiment analysis using TensorFlow/Keras.

## Table of Contents
1. [Setup and Data Generation](#setup)
2. [Data Preprocessing](#preprocessing)
3. [Model Architecture Comparison](#models)
4. [Training and Evaluation](#training)
5. [Model Interpretation](#interpretation)
6. [Inference Examples](#inference)

## 1. Setup and Data Generation <a id='setup'></a>

In [ ]:
# Import libraries
import sys
sys.path.append('..')  # Add parent directory to path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from model import create_model
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Generate synthetic sentiment data
def generate_data(num_samples=5000):
    """Generate synthetic sentiment data"""
    positive_words = ['excellent', 'amazing', 'wonderful', 'fantastic', 'great', 'love',
                     'perfect', 'best', 'awesome', 'outstanding', 'brilliant', 'superb']
    
    negative_words = ['terrible', 'awful', 'horrible', 'worst', 'bad', 'hate',
                     'disappointing', 'poor', 'useless', 'waste', 'broken', 'defective']
    
    neutral_words = ['okay', 'average', 'normal', 'standard', 'typical', 'regular',
                    'acceptable', 'fine', 'moderate', 'adequate', 'fair', 'decent']
    
    contexts = ['product', 'service', 'item', 'purchase', 'experience', 'quality']
    
    texts = []
    labels = []
    
    for _ in range(num_samples // 3):
        # Positive
        texts.append(f"This {np.random.choice(contexts)} is {np.random.choice(positive_words)}!")
        labels.append(1)
        
        # Negative
        texts.append(f"This {np.random.choice(contexts)} is {np.random.choice(negative_words)}.")
        labels.append(0)
        
        # Neutral
        texts.append(f"This {np.random.choice(contexts)} is {np.random.choice(neutral_words)}.")
        labels.append(2)
    
    return texts, labels

texts, labels = generate_data(5000)
print(f"Generated {len(texts)} samples")
print(f"\nSample texts:")
for i in range(5):
    print(f"{i+1}. {texts[i]} - Label: {labels[i]}")

In [ ]:
# Create DataFrame
df = pd.DataFrame({
    'text': texts,
    'sentiment': labels
})

# Map labels to names
sentiment_map = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
df['sentiment_name'] = df['sentiment'].map(sentiment_map)

print("Dataset Info:")
print(df.info())
print("\nSentiment Distribution:")
print(df['sentiment_name'].value_counts())

# Visualize distribution
plt.figure(figsize=(10, 6))
df['sentiment_name'].value_counts().plot(kind='bar', color=['coral', 'lightgreen', 'lightblue'])
plt.title('Sentiment Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 2. Data Preprocessing <a id='preprocessing'></a>

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values, df['sentiment'].values, 
    test_size=0.2, random_state=42, stratify=df['sentiment']
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.2, random_state=42, stratify=y_train
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Text statistics
text_lengths = [len(text.split()) for text in X_train]

print("Text Length Statistics:")
print(f"Mean: {np.mean(text_lengths):.2f}")
print(f"Median: {np.median(text_lengths):.2f}")
print(f"Max: {np.max(text_lengths)}")
print(f"Min: {np.min(text_lengths)}")

plt.figure(figsize=(10, 6))
plt.hist(text_lengths, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(np.mean(text_lengths), color='r', linestyle='--', label=f'Mean: {np.mean(text_lengths):.2f}')
plt.title('Distribution of Text Lengths', fontsize=14, fontweight='bold')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Tokenization
vocab_size = 5000
max_length = 50

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert to sequences
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), 
                            maxlen=max_length, padding='post', truncating='post')
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(X_val), 
                         maxlen=max_length, padding='post', truncating='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), 
                          maxlen=max_length, padding='post', truncating='post')

print(f"Vocabulary size: {len(tokenizer.word_index)}")
print(f"Sequence shape: {X_train_seq.shape}")
print(f"\nExample sequence:")
print(f"Text: {X_train[0]}")
print(f"Sequence: {X_train_seq[0][:10]}...")

## 3. Model Architecture Comparison <a id='models'></a>

In [ ]:
# Create and compare different models
models_to_compare = ['dense', 'lstm', 'gru', 'cnn_lstm']
results = {}

for model_type in models_to_compare:
    print(f"\n{'='*60}")
    print(f"Training {model_type.upper()} Model")
    print(f"{'='*60}")
    
    # Create model
    model = create_model(
        model_type=model_type,
        vocab_size=vocab_size,
        embedding_dim=64,
        max_length=max_length,
        num_classes=3
    )
    
    # Early stopping
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
    
    # Train
    history = model.model.fit(
        X_train_seq, y_train,
        validation_data=(X_val_seq, y_val),
        epochs=10,
        batch_size=32,
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Evaluate
    test_loss, test_acc = model.model.evaluate(X_test_seq, y_test, verbose=0)
    
    # Store results
    results[model_type] = {
        'history': history.history,
        'test_loss': test_loss,
        'test_accuracy': test_acc,
        'model': model
    }
    
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Compare model performances
comparison_df = pd.DataFrame({
    'Model': [m.upper() for m in results.keys()],
    'Test Accuracy': [results[m]['test_accuracy'] for m in results.keys()],
    'Test Loss': [results[m]['test_loss'] for m in results.keys()]
})

print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(comparison_df['Model'], comparison_df['Test Accuracy'], 
           color='steelblue', edgecolor='black')
axes[0].set_title('Model Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Test Accuracy')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(comparison_df['Model'], comparison_df['Test Loss'], 
           color='coral', edgecolor='black')
axes[1].set_title('Model Loss Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Test Loss')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. Training and Evaluation <a id='training'></a>

In [ ]:
# Select best model
best_model_type = max(results.keys(), key=lambda k: results[k]['test_accuracy'])
best_model = results[best_model_type]['model']

print(f"Best Model: {best_model_type.upper()}")
print(f"Test Accuracy: {results[best_model_type]['test_accuracy']:.4f}")

In [ ]:
# Plot training history for best model
history = results[best_model_type]['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history['accuracy'], label='Training')
axes[0].plot(history['val_accuracy'], label='Validation')
axes[0].set_title(f'{best_model_type.upper()} Model Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history['loss'], label='Training')
axes[1].plot(history['val_loss'], label='Validation')
axes[1].set_title(f'{best_model_type.upper()} Model Loss', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Predictions and classification report
y_pred_probs = best_model.model.predict(X_test_seq)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred, 
                           target_names=['Negative', 'Positive', 'Neutral']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Negative', 'Positive', 'Neutral'],
           yticklabels=['Negative', 'Positive', 'Neutral'])
plt.title(f'Confusion Matrix - {best_model_type.upper()} Model', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 5. Model Interpretation <a id='interpretation'></a>

In [ ]:
# Analyze prediction confidence
max_probs = np.max(y_pred_probs, axis=1)
correct_predictions = (y_pred == y_test)

print("Prediction Confidence Analysis:")
print(f"Mean confidence (correct): {np.mean(max_probs[correct_predictions]):.4f}")
print(f"Mean confidence (incorrect): {np.mean(max_probs[~correct_predictions]):.4f}")

# Plot confidence distribution
plt.figure(figsize=(10, 6))
plt.hist(max_probs[correct_predictions], bins=30, alpha=0.5, label='Correct', color='green')
plt.hist(max_probs[~correct_predictions], bins=30, alpha=0.5, label='Incorrect', color='red')
plt.title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Confidence (Max Probability)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Show examples of correct and incorrect predictions
print("\nExamples of Correct Predictions:")
correct_indices = np.where(correct_predictions)[0][:5]
for idx in correct_indices:
    print(f"\nText: {X_test[idx]}")
    print(f"True: {sentiment_map[y_test[idx]]} | Predicted: {sentiment_map[y_pred[idx]]}")
    print(f"Confidence: {max_probs[idx]:.4f}")

print("\n" + "="*70)
print("Examples of Incorrect Predictions:")
incorrect_indices = np.where(~correct_predictions)[0][:5]
for idx in incorrect_indices:
    print(f"\nText: {X_test[idx]}")
    print(f"True: {sentiment_map[y_test[idx]]} | Predicted: {sentiment_map[y_pred[idx]]}")
    print(f"Confidence: {max_probs[idx]:.4f}")

## 6. Inference Examples <a id='inference'></a>

In [ ]:
# Function to predict sentiment
def predict_sentiment(text, model, tokenizer):
    """Predict sentiment for a given text"""
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')
    prediction = model.model.predict(padded, verbose=0)
    predicted_class = np.argmax(prediction[0])
    
    return {
        'sentiment': sentiment_map[predicted_class],
        'confidence': prediction[0][predicted_class],
        'probabilities': {
            'Negative': prediction[0][0],
            'Positive': prediction[0][1],
            'Neutral': prediction[0][2]
        }
    }

# Test with custom examples
test_examples = [
    "This product is absolutely amazing!",
    "Terrible experience, very disappointed.",
    "It's okay, nothing special.",
    "Best purchase I've ever made!",
    "Waste of money, would not recommend.",
    "The quality is acceptable for the price."
]

print("Custom Sentiment Predictions:")
print("="*70)

for text in test_examples:
    result = predict_sentiment(text, best_model, tokenizer)
    print(f"\nText: '{text}'")
    print(f"Predicted Sentiment: {result['sentiment']} (Confidence: {result['confidence']:.4f})")
    print(f"Probabilities: Neg={result['probabilities']['Negative']:.3f}, "
          f"Pos={result['probabilities']['Positive']:.3f}, "
          f"Neu={result['probabilities']['Neutral']:.3f}")

In [ ]:
# Interactive prediction (if running in interactive environment)
print("\n" + "="*70)
print("Try your own text!")
print("="*70)

# Example of how to use in production
def analyze_sentiment(text):
    """Production-ready sentiment analysis function"""
    result = predict_sentiment(text, best_model, tokenizer)
    return result

# Example usage
sample_text = "The service exceeded my expectations!"
analysis = analyze_sentiment(sample_text)

print(f"\nSample Analysis:")
print(f"Input: '{sample_text}'")
print(f"Sentiment: {analysis['sentiment']}")
print(f"Confidence: {analysis['confidence']:.2%}")

## Summary

This notebook demonstrated:

1. **Data Preparation**: Generated and preprocessed sentiment data
2. **Model Architectures**: Compared Dense, LSTM, GRU, and CNN-LSTM models
3. **Training**: Trained models with early stopping and validation
4. **Evaluation**: Assessed performance with metrics and visualizations
5. **Interpretation**: Analyzed prediction confidence and examples
6. **Inference**: Created production-ready prediction function

### Key Takeaways:
- Neural networks can effectively classify text sentiment
- Different architectures have trade-offs (speed vs accuracy)
- Proper preprocessing is crucial for NLP tasks
- Model confidence provides valuable insights
- The approach scales to real-world applications

### Next Steps:
- Use real-world datasets (IMDB, Twitter, product reviews)
- Implement attention mechanisms
- Fine-tune pre-trained models (BERT, RoBERTa)
- Deploy as REST API or web service
- Add multi-language support